Accès Aux données

In [ ]:
!pip install -q gdown

# Télécharger tout le dossier Google Drive
!gdown --folder "https://drive.google.com/drive/folders/1oHCdRPWimLvTzcc8sYbCgnB61sLcCMZo?usp=sharing"

PARTIE I: Bibliothèques Python pour l'nalyse spatiale et la cartographie





In [ ]:
# ─────────────────────────────────────────────
# Installation des bibliothèques (Colab)
# ─────────────────────────────────────────────
!pip install geopandas folium plotly mapclassify contextily pyogrio shapely --quiet
!pip install keplergl ipyleaflet branca --quiet

In [ ]:
# ─────────────────────────────────────────────
# Importation des bibliothèques
# ─────────────────────────────────────────────
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium import plugins
from folium.plugins import TimestampedGeoJson, HeatMap, MarkerCluster, MiniMap, MeasureControl
import contextily as ctx
from shapely.geometry import box, Point
import warnings
warnings.filterwarnings('ignore')



In [ ]:
pollution_geojson_path = '/content/Rendu_Bridier/Pollutions_ann/Pollution.geojson'
df_raw_pollution = gpd.read_file(pollution_geojson_path)

In [ ]:
print("Unique dates in df_raw_pollution['date_debut'] and their counts:")
display(df_raw_pollution['date_debut'].value_counts().sort_index())

Limites Folium pour le chargement de couches(Shp et Choroplèthes)

In [ ]:
# ─────────────────────────────────────────────
# Démonstration des limites de Folium
# ─────────────────────────────────────────────

limites_folium = {
    "Problème": [
        "Taille du fichier HTML",
        "Rendu navigateur",
        "Géométries complexes",
        "Données raster",
        "Mise à jour dynamique",
        "Performance > 50k polygones",
        "Tooltips sur gros GeoJSON"
    ],
    "Description": [
        "GeoJSON intégré → HTML > 100 MB possible avec données nationales",
        "Tous les objets chargés en mémoire JS → ralentissement sévère",
        "Polygones à haute résolution → freeze navigateur",
        "Pas de support natif des rasters GeoTIFF",
        "Pas de mise à jour des couches sans rechargement",
        "Dépasse 500 MB RAM → crash",
        "Latence tooltip > 1s avec > 10k features"
    ],
    "Solution": [
        "Simplifier avec .simplify() GeoPandas ou utiliser KeplerGL",
        "Filtrer / agréger les données avant export",
        "Douglas-Peucker tolerance=0.001 dans .simplify()",
        "Convertir en PNG et utiliser ImageOverlay",
        "Préférer ipyleaflet ou Plotly Dash",
        "Passer à DeckGL ou Mapbox GL JS",
        "Réduire les colonnes exportées dans le GeoJSON"
    ]
}

df_limites = pd.DataFrame(limites_folium)
print("⚠️  LIMITES DE FOLIUM POUR LE CHARGEMENT DE COUCHES")
print("="*70)
for _, row in df_limites.iterrows():
    print(f"\n🔴 {row['Problème']}")
    print(f"   Problème : {row['Description']}")
    print(f"   Solution : {row['Solution']}")

Exemples d'utilisation Geopandas

In [ ]:
commune_shp_path = '/content/Rendu_Bridier/Limites_Com_Metr/commune-bd-topo.shp'
zfe_geojson_path = '/content/Rendu_Bridier/ZFE/zfe_zone_metropole-aix-marseille-provence.geojson'

In [ ]:
# ─────────────────────────────────────────────
# Exemple d'utilisation GeoPandas : opérations spatiales de base
# ─────────────────────────────────────────────


# Charger les données pour les communes
gdf_communes = gpd.read_file(commune_shp_path)

# Remove duplicaty
# This is necessary because GeoJSON conversion requires unique column names
gdf_communes = gdf_communes.loc[:, ~gdf_communes.columns.duplicated()]

# Charger les données pour le ZFE
gdf_zfe = gpd.read_file(zfe_geojson_path)

print("✅ GeoDataFrame communes chargé :")
# Display relevant columns from the real data
print(gdf_communes[['Nom_officie', 'codeinsee']].head())
print(f"\n   CRS : {gdf_communes.crs}")
print(f"   Nombre de communes : {len(gdf_communes)}")
print(f"   Étendue (bbox) : {gdf_communes.total_bounds}")

print("\n✅ GeoDataFrame ZFE chargé :")
print(gdf_zfe.head())
print(f"\n   CRS : {gdf_zfe.crs}")
print(f"   Nombre de zones ZFE : {len(gdf_zfe)}")
print(f"   Étendue (bbox) : {gdf_zfe.total_bounds}")

In [ ]:
# ─────────────────────────────────────────────
# GeoPandas : reprojection, buffer, intersection
# ─────────────────────────────────────────────

# 1. Reprojection en Lambert-93 (EPSG:2154) pour calculs métriques
gdf_communes_L93  = gdf_communes.to_crs(epsg=2154)
gdf_zfe_L93       = gdf_zfe.to_crs(epsg=2154)

# 2. Aire de chaque commune (km²)
gdf_communes_L93["aire_km2"] = gdf_communes_L93.geometry.area / 1e6

# 3. Buffer de 2 km autour de la ZFE
zfe_buffer_2km = gdf_zfe_L93.copy()
zfe_buffer_2km["geometry"] = gdf_zfe_L93.geometry.buffer(2000)

# 4. Simplification géométrique (utile avant export Folium)
gdf_communes_simplified = gdf_communes.copy()
gdf_communes_simplified["geometry"] = gdf_communes.geometry.simplify(tolerance=0.001)

# 5. Intersection : communes qui intersectent la ZFE
communes_dans_zfe = gdf_communes_L93[gdf_communes_L93.intersects(gdf_zfe_L93.union_all())]

print("✅ Opérations spatiales effectuées :")
print(f"   Communes intersectant la ZFE : {communes_dans_zfe['Nom_officie'].tolist()}")
print(f"   Superficie ZFE (buffer 2km inclus) : {zfe_buffer_2km.to_crs(epsg=2154).geometry.area.sum()/1e6:.2f} km²")
print(f"\n   Aires des communes (km²) :")
print(gdf_communes_L93[["Nom_officie", "aire_km2"]].sort_values("aire_km2", ascending=False).to_string(index=False))
##%% md
## 1.4 Exemples d'utilisation — Folium (carte interactive)

Exemple d'utilisation Folium

In [ ]:
 # ─────────────────────────────────────────────
# Exemple d'utilisation Folium : carte de base avec couches multiples
# ─────────────────────────────────────────────

# Centre de la métropole AMP
centre_AMP = [43.40, 5.20]

m = folium.Map(
    location=centre_AMP,
    zoom_start=10,
    tiles=None  # on va ajouter les fonds manuellement
)

# ── Fonds de carte disponibles ──
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="CartoDB", name="CartoDB Positron"
).add_to(m)
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png",
    attr="CartoDB", name="CartoDB Dark"
).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satellite"
).add_to(m)

# Convert datetime columns to string to avoid TypeError during JSON serialization for Folium
gdf_zfe_for_map = gdf_zfe.copy()
for col in ['date_debut', 'date_fin']:
    if col in gdf_zfe_for_map.columns:
        gdf_zfe_for_map[col] = gdf_zfe_for_map[col].astype(str)

zfe_buffer_2km_for_map = zfe_buffer_2km.to_crs(epsg=4326).copy()
for col in zfe_buffer_2km_for_map.select_dtypes(include=['datetime64[ns, UTC]', 'datetime64[ns]']).columns:
    zfe_buffer_2km_for_map[col] = zfe_buffer_2km_for_map[col].astype(str)

# ── Couche communes ──
folium.GeoJson(
    gdf_communes_simplified.__geo_interface__,
    name="Communes AMP",
    style_function=lambda x: {
        "fillColor": "#3B6DE1",
        "color": "white",
        "weight": 1.5,
        "fillOpacity": 0.25
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["Nom_officie", "codeinsee"],
        aliases=["Commune", "Code INSEE"],
        localize=True
    )
).add_to(m)

# ── Couche ZFE ──
folium.GeoJson(
    gdf_zfe_for_map.__geo_interface__,
    name="ZFE Marseille",
    style_function=lambda x: {
        "fillColor": "#E53935",
        "color": "#B71C1C",
        "weight": 3,
        "fillOpacity": 0.35,
        "dashArray": "5, 5"
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['id', 'date_debut', 'vp_critair'],
        aliases=['ID ZFE:', 'Début:', 'Critair VP:'],
        localize=True
    )
).add_to(m)

# ── Buffer ZFE reprojété en WGS84 pour Folium ──
folium.GeoJson(
    zfe_buffer_2km_for_map.__geo_interface__,
    name="Buffer 2km ZFE",
    style_function=lambda x: {
        "fillColor": "#FF9800",
        "color": "#E65100",
        "weight": 2,
        "fillOpacity": 0.15,
        "dashArray": "8, 4"
    }
).add_to(m)

# ── Plugins ──
MiniMap(toggle_display=True).add_to(m)
MeasureControl(position="bottomleft", primary_length_unit="kilometers").add_to(m)
plugins.Fullscreen(position="topright").add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

# Légende HTML
legend_html = """
<div style="position: fixed; bottom: 50px; right: 10px; z-index: 1000;
     background-color: white; padding: 12px; border-radius: 8px;
     border: 2px solid #ccc; font-family: Arial; font-size: 13px;">
  <b>Légende</b><br>
  <span style="background:#3B6DE1; width:14px; height:14px; display:inline-block; opacity:0.5;"></span>&nbsp;Communes AMP<br>
  <span style="background:#E53935; width:14px; height:14px; display:inline-block; opacity:0.5;"></span>&nbsp;ZFE Marseille<br>
  <span style="background:#FF9800; width:14px; height:14px; display:inline-block; opacity:0.4;"></span>&nbsp;Buffer 2km<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m

Exemple d'utilisation Plotly express

In [ ]:
import matplotlib.colors as mcolors
import datetime # Import datetime
import pandas as pd # Ensure pandas is imported for CSV reading
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import geopandas as gpd # Import geopandas for reading GeoJSON

# ─────────────────────────────────────────────
# Exemple d'utilisation Plotly : série temporelle de pollution (données réelles)
# ─────────────────────────────────────────────

# Charger les données réelles depuis le fichier Pollution.geojson
pollution_geojson_path = '/content/Rendu_Bridier/Pollutions_ann/Pollution.geojson'
df_raw_pollution = gpd.read_file(pollution_geojson_path) # Changed to gpd.read_file

# --- Debugging: Print columns to identify correct date column ---
print("Columns in df_raw_pollution:", df_raw_pollution.columns)

# Directly create the 'date' column from 'date_debut' and convert to datetime
# Corrected column name from 'Date de début' to 'date_debut'
df_raw_pollution['date'] = pd.to_datetime(df_raw_pollution['date_debut'])

# Pivot the data to create columns for each pollutant
# We'll use 'valeur' as the primary value for plotting, and 'label_polluant' for columns
df_pollution = df_raw_pollution.pivot_table(
    index='date',
    columns='label_polluant',
    values='valeur',
    aggfunc='mean' # Use mean if there are multiple readings for the same date/pollutant
).reset_index()

# Rename columns to match the expected format (e.g., NO2_ugm3)
# This assumes the 'label_polluant' column contains values like 'NO2', 'PM10', etc.
# Only rename if the column is not 'date' (which is already correctly named)
current_cols = df_pollution.columns.tolist()
new_cols = []
for col in current_cols:
    # Use 'label_polluant' from df_raw_pollution to get unique pollutant labels
    if col != 'date' and col in df_raw_pollution['label_polluant'].unique(): # Check if it's a pollutant column
        new_cols.append(f'{col}_ugm3')
    else:
        new_cols.append(col)
df_pollution.columns = new_cols

# Afficher les premières lignes et les types de données pour vérification
print("✅ Données de pollution réelles chargées et préparées pour le graphique :")
display(df_pollution.head())
print(df_pollution.info())

# ── Graphique Plotly interactif ──
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=["NO₂ (μg/m³)", "PM10 (μg/m³)", "PM2.5 (μg/m³)", "O₃ (μg/m³)"],
    shared_xaxes=True
)

# Updated polluants list with actual column names from df_pollution and display names
polluants_data = [
    ("Dioxyde d'azote (gaz)_ugm3", "NO₂", "#E53935", 1, 1),
    ("Particules en suspension <10 µm (masses) (PM10)_ugm3", "PM10", "#F4511E", 1, 2),
    ("Particules en suspension <2.5 µm (masses) (PM2.5)_ugm3", "PM2.5", "#FB8C00", 2, 1),
    ("Ozone (gaz)_ugm3", "O₃", "#43A047", 2, 2)
]

for col_name, display_name, color, row, col_idx in polluants_data:
    # Convert hex color to rgba for fillcolor
    rgb_tuple = mcolors.to_rgb(color)
    rgba_fillcolor = f"rgba({int(255*rgb_tuple[0])},{int(255*rgb_tuple[1])},{int(255*rgb_tuple[2])},0.1)"

    # Check if the pollutant column exists in df_pollution before plotting
    if col_name in df_pollution.columns:
        fig.add_trace(
            go.Scatter(x=df_pollution["date"], y=df_pollution[col_name],
                       mode="lines", name=display_name,
                       line=dict(color=color, width=1.5),
                       fill="tozeroy", fillcolor=rgba_fillcolor),
            row=row, col=col_idx
        )
    else:
        print(f"⚠️ Column '{col_name}' not found in df_pollution. Skipping plot for {display_name}.")

    # Ligne verticale ZFE
    # Convert datetime object to milliseconds since epoch for better compatibility with Plotly's internal date handling
    zfe_date_dt = datetime.datetime(2021, 1, 1)
    zfe_date_timestamp_ms = zfe_date_dt.timestamp() * 1000 # Convert to milliseconds
    fig.add_vline(x=zfe_date_timestamp_ms, line_dash="dash", line_color="navy",
                  annotation_text="ZFE" if row==1 and col_idx==1 else "",
                  row=row, col=col_idx)

fig.update_layout(
    title="Évolution de la pollution atmosphérique — Métropole AMP (2016–2025)",
    height=500, template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.25),
    font=dict(family="Helvetica, Arial", size=12)
)
fig.show()

In [ ]:
# ─────────────────────────────────────────────
# Plotly : Heatmap saisonnalité (mois × année)
# ─────────────────────────────────────────────

# Prepending df_poll creation for the Plotly Heatmap to resolve NameError.
# This code is moved from a later cell (`dde3d038`) to ensure df_poll is defined.
# Ensure 'date' column is datetime and 'inseecom' is string for consistency
df_raw_pollution['date'] = pd.to_datetime(df_raw_pollution['date_debut'])
df_raw_pollution['inseecom'] = df_raw_pollution['inseecom'].astype(str)

df_poll_grouped = df_raw_pollution.groupby(['date', 'inseecom', 'label_polluant'])['valeur'].mean().reset_index()

df_poll = df_poll_grouped.pivot_table(
    index=['date', 'inseecom'],
    columns='label_polluant',
    values='valeur'
).reset_index()

df_poll = df_poll.rename(columns={
    "Dioxyde d'azote (gaz)": "NO2_ugm3",
    "Particules en suspension <10 µm (masses) (PM10)": "PM10_ugm3",
    "Particules en suspension <2.5 µm (masses) (PM2.5)": "PM25_ugm3",
    "Ozone (gaz)": "O3_ugm3"
})

# Add 'dans_zfe' column (assuming 'communes_dans_zfe' is defined earlier)
zfe_communes_insee = communes_dans_zfe['codeinsee'].unique().tolist()
df_poll['dans_zfe'] = df_poll['inseecom'].isin(zfe_communes_insee).astype(int)

# Original code for heatmap generation starts here
df_poll["annee"] = df_poll["date"].dt.year
df_poll["mois"] = df_poll["date"].dt.month

# Aggregate NO2_ugm3 by annee and mois before pivoting to ensure unique index/column combinations
pivot_data_agg = df_poll.groupby(["annee", "mois"])["NO2_ugm3"].mean().reset_index()
pivot_no2 = pivot_data_agg.pivot(index="annee", columns="mois", values="NO2_ugm3")

mois_labels = ["Jan","Fév","Mar","Avr","Mai","Jun","Jul","Aoû","Sep","Oct","Nov","Déc"]

fig_heatmap = go.Figure(go.Heatmap(
    z=pivot_no2.values,
    x=mois_labels,
    y=pivot_no2.index,
    colorscale="RdYlGn_r",
    colorbar=dict(title="NO₂ (μg/m³)"),
    hoverongaps=False,
    text=pivot_no2.values.round(1),
    texttemplate="%{text}"
))

fig_heatmap.update_layout(
    title="Heatmap NO₂ — Saisonnalité mensuelle par année (Métropole AMP)",
    xaxis_title="Mois", yaxis_title="Année",
    template="plotly_white", height=400,
    shapes=[dict(type="rect", x0=-0.5, x1=11.5, y0=2020.5, y1=2021.5,
                 fillcolor="rgba(0,0,255,0.08)", line=dict(color="navy", width=2))]
)
fig_heatmap.add_annotation(
    x=11.5, y=2021, text="Mise en place ZFE",
    showarrow=False, font=dict(color="navy", size=10)
)
fig_heatmap.show()

In [ ]:
print(df_raw_pollution.info())

PARTIE II: Mise en pratique: Analyse spatiale de la ZFE de Marseille

In [ ]:
# ─────────────────────────────────────────────
# Folium : carte choroplèthe avec seuils pollution
# ─────────────────────────────────────────────

# --- 1. Préparer les données de pollution par commune ---
# Agrégation des données de pollution brutes par commune (inseecom) et par type de polluant
# Calcul de la moyenne des 'valeur' pour chaque polluant dans chaque commune
df_pollution_commune = df_raw_pollution.groupby(['inseecom', 'label_polluant'])['valeur'].mean().unstack().reset_index()

# Renommer les colonnes des polluants pour une meilleure lisibilité
# Assurez-vous que ces noms correspondent aux valeurs réelles de 'label_polluant' dans votre CSV
df_pollution_commune = df_pollution_commune.rename(columns={
    "Dioxyde d'azote (gaz)": "NO2_moyen",
    "Particules en suspension <10 µm (masses) (PM10)": "PM10_moyen",
    "Particules en suspension <2.5 µm (masses) (PM2.5)": "PM25_moyen",
    "Ozone (gaz)": "O3_moyen"
})

# Convert 'inseecom' to string type to match 'codeinsee' in gdf_communes for merging
df_pollution_commune['inseecom'] = df_pollution_commune['inseecom'].astype(str)

# Fusionner les données de pollution agrégées avec le GeoDataFrame des communes
# Utilisation de 'codeinsee' de gdf_communes et 'inseecom' de df_pollution_commune comme clés
gdf_communes_pollution = gdf_communes.merge(df_pollution_commune, left_on='codeinsee', right_on='inseecom', how='left')

# Supprimer la colonne 'inseecom' redondante après la fusion
if 'inseecom' in gdf_communes_pollution.columns:
    gdf_communes_pollution = gdf_communes_pollution.drop(columns=['inseecom'])

# Gérer les valeurs NaN qui pourraient apparaître pour les communes sans données de pollution
# en les remplissant avec 0 (ou une autre stratégie appropriée comme la moyenne/médiane)
for col in ['NO2_moyen', 'PM10_moyen', 'PM25_moyen', 'O3_moyen']:
    if col in gdf_communes_pollution.columns:
        gdf_communes_pollution[col] = gdf_communes_pollution[col].fillna(0)

print("✅ GeoDataFrame communes avec données de pollution fusionnées :")
display(gdf_communes_pollution.head())

# --- 2. Carte choroplèthe Folium ---
# Centre de la métropole AMP
centre_AMP = [43.40, 5.20]

m2 = folium.Map(location=centre_AMP, zoom_start=10,
                tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
                attr="CartoDB Positron")

# Choroplèthe NO2 avec tooltips intégrés pour éviter la surcharge d'étiquettes
folium.Choropleth(
    geo_data=gdf_communes_pollution.__geo_interface__,
    data=gdf_communes_pollution,
    columns=["codeinsee", "NO2_moyen"], # Utilise codeinsee pour la clé et NO2_moyen pour les valeurs
    key_on="feature.properties.codeinsee", # Assurez-vous que cela correspond à la propriété du GeoJSON
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.4,
    legend_name="NO₂ moyen (μg/m³)",
    name="Pollution NO₂",
    bins=5,
    tooltip=folium.GeoJsonTooltip( # Tooltip directement sur la couche choroplèthe
        fields=["Nom_officie", "NO2_moyen", "PM10_moyen", "PM25_moyen", "O3_moyen"], # Champs à afficher
        aliases=["Commune", "NO₂ moy. (μg/m³)", "PM10 moy. (μg/m³)", "PM2.5 moy. (μg/m³)", "O₃ moy. (μg/m³)"],
        localize=True,
        style="font-family: Arial; font-size: 12px;"
    )
).add_to(m2)

# ZFE par dessus (réutilise gdf_zfe_for_map pour la gestion des dates)
folium.GeoJson(
    gdf_zfe_for_map.__geo_interface__,
    name="ZFE Marseille",
    style_function=lambda x: {
        "fillColor": "none",
        "color": "#1a1aff",
        "weight": 3.5,
        "dashArray": "6, 3"
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['id', 'date_debut', 'vp_critair'],
        aliases=['ID ZFE:', 'Début:', 'Critair VP:'],
        localize=True
    )
).add_to(m2)

MiniMap().add_to(m2)
folium.LayerControl().add_to(m2)
m2

Exploration Initiale des Données

In [ ]:
print("=" * 55)
print("  EXPLORATION DES DONNÉES GÉOGRAPHIQUES")
print("=" * 55)

for name, gdf in [("Communes", gdf_communes), ("ZFE", gdf_zfe)]:
    print(f"\n📂 {name}")
    print(f"   Shape    : {gdf.shape}")
    print(f"   CRS      : {gdf.crs}")
    print(f"   Colonnes : {list(gdf.columns)}")
    print(f"   Étendue  : {gdf.total_bounds.round(4)}")

print("\n" + "=" * 55)
print("  EXPLORATION DES DONNÉES DE POLLUTION")
print("=" * 55)

print("\n📂 Pollution (Données brutes)")
print(f"   Shape    : {df_raw_pollution.shape}")
print(f"   Colonnes : {list(df_raw_pollution.columns)}")
print(f"   Types    :\n{df_raw_pollution.info()}")

 Nettoyage et préparation des données de pollution

In [ ]:
# --- 1. Préparer les données de pollution pour l'analyse temporelle et par commune ---
# Agrégation des données de pollution brutes par date, commune (inseecom) et par type de polluant
# Calcul de la moyenne des 'valeur' pour chaque polluant dans chaque commune et par jour

# Ensure 'date' column is datetime and 'inseecom' is string for consistency
df_raw_pollution['date'] = pd.to_datetime(df_raw_pollution['date_debut'])
df_raw_pollution['inseecom'] = df_raw_pollution['inseecom'].astype(str)

df_poll_grouped = df_raw_pollution.groupby(['date', 'inseecom', 'label_polluant'])['valeur'].mean().reset_index()

# Pivot the data to have pollutant types as columns
df_poll = df_poll_grouped.pivot_table(
    index=['date', 'inseecom'],
    columns='label_polluant',
    values='valeur'
).reset_index()

# Rename columns for consistency and easier access
df_poll = df_poll.rename(columns={
    "Dioxyde d'azote (gaz)": "NO2_ugm3",
    "Particules en suspension <10 µm (masses) (PM10)": "PM10_ugm3",
    "Particules en suspension <2.5 µm (masses) (PM2.5)": "PM25_ugm3",
    "Ozone (gaz)": "O3_ugm3"
})

# Add 'annee' and 'mois' columns
df_poll["annee"] = df_poll["date"].dt.year
df_poll["mois"] = df_poll["date"].dt.month

# Add 'dans_zfe' column
# The 'communes_dans_zfe' GeoDataFrame has 'codeinsee' for communes intersecting ZFE.
# We need to create a list of these INSEE codes.
zfe_communes_insee = communes_dans_zfe['codeinsee'].unique().tolist()
df_poll['dans_zfe'] = df_poll['inseecom'].isin(zfe_communes_insee).astype(int)

# Display initial exploration results, similar to the synthetic data block
print(f"✅ Données de pollution réelles prêtes : {df_poll.shape[0]:,} observations")
print(f"   Communes : {df_poll['inseecom'].nunique()}")
print(f"   Période  : {df_poll['date'].min().strftime('%Y-%m')} → {df_poll['date'].max().strftime('%Y-%m')}")
print(f"\n   Statistiques descriptives :")
display(df_poll[["NO2_ugm3","PM10_ugm3","PM25_ugm3","O3_ugm3"]].describe().round(2))

 Analyse des Tendances de Pollution Annuelle

In [ ]:
df_annuel = df_poll.groupby(["annee", "dans_zfe"])[["NO2_ugm3","PM10_ugm3","PM25_ugm3","O3_ugm3"]].mean().reset_index()

df_zfe     = df_annuel[df_annuel["dans_zfe"]==1]
df_hors    = df_annuel[df_annuel["dans_zfe"]==0]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Évolution annuelle des polluants atmosphériques\nZFE Marseille vs Communes hors ZFE — 2016–2025",
             fontsize=14, fontweight="bold", y=1.01)

polluants_info = [
    ("NO2_ugm3",  "NO₂ (μg/m³)",   "Valeur limite OMS : 40 μg/m³",  40, "#E53935"),
    ("PM10_ugm3", "PM10 (μg/m³)",  "Valeur limite OMS : 20 μg/m³",  20, "#F4511E"),
    ("PM25_ugm3", "PM2.5 (μg/m³)", "Valeur limite OMS : 10 μg/m³",  10, "#FB8C00"),
    ("O3_ugm3",   "O₃ (μg/m³)",    "Valeur cible : 120 μg/m³",     120, "#43A047"),
]

for ax, (col, ylabel, label_oms, seuil, color) in zip(axes.flat, polluants_info):
    # Check if data for ZFE exists for the pollutant
    if not df_zfe.empty and col in df_zfe.columns:
        ax.plot(df_zfe["annee"], df_zfe[col], "o-", color=color, lw=2.5, ms=5, label="ZFE (Marseille)")
    else:
        print(f"Warning: No ZFE data for {col}")

    # Check if data for non-ZFE exists for the pollutant
    if not df_hors.empty and col in df_hors.columns:
        ax.plot(df_hors["annee"], df_hors[col], "s--", color="steelblue", lw=2, ms=4, label="Hors ZFE")
    else:
        print(f"Warning: No 'Hors ZFE' data for {col}")

    ax.axvline(x=2021, color="navy", lw=1.8, ls=":", label="Création ZFE")
    ax.axhline(y=seuil, color="gray", lw=1.2, ls="--", alpha=0.6, label=label_oms)

    # Ensure ax.get_ylim() returns valid numbers for fill_between
    current_ylim = ax.get_ylim()
    y_min_fill = max(0, current_ylim[0]) if current_ylim[0] is not None else 0
    y_max_fill = current_ylim[1] if current_ylim[1] is not None else 100 # Default max if none

    ax.fill_between([2021, 2025], y_min_fill, y_max_fill, alpha=0.05, color="navy")
    ax.set_title(ylabel, fontweight="bold")
    ax.set_xlabel("Année")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(2016, 2025)

plt.tight_layout()
plt.savefig("evolution_pollution_2016_2025.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Graphique sauvegardé : evolution_pollution_2016_2025.png")

 Test Statistique Avant/Après ZFE

In [ ]:
from scipy import stats

df_avant = df_poll[(df_poll["annee"] < 2021)  & (df_poll["dans_zfe"]==1)]
df_apres = df_poll[(df_poll["annee"] >= 2021) & (df_poll["dans_zfe"]==1)]

print("=" * 65)
print("  TEST STATISTIQUE — Mann-Whitney U (avant/après ZFE 2021)")
print("  Zone : Marseille (dans ZFE)")
print("=" * 65)

resultats = []
for polluant in ["NO2_ugm3", "PM10_ugm3", "PM25_ugm3", "O3_ugm3"]:
    avant = df_avant[polluant].dropna()
    apres = df_apres[polluant].dropna()

    # Only proceed if both 'avant' and 'apres' have data
    if not avant.empty and not apres.empty:
        stat, pval = stats.mannwhitneyu(avant, apres, alternative="greater")
        moy_avant = avant.mean()
        moy_apres = apres.mean()
        variation = (moy_apres - moy_avant) / moy_avant * 100
        sig = "✅ Significatif" if pval < 0.05 else "❌ Non significatif"
        resultats.append({
            "Polluant": polluant.replace("_ugm3",""),
            "Moy. avant": round(moy_avant, 2),
            "Moy. après": round(moy_apres, 2),
            "Variation %": round(variation, 1),
            "p-valeur": round(pval, 4),
            "Résultat": sig
        })
    else:
        print(f"Warning: Not enough data for {polluant} to perform statistical test (before and/or after ZFE). Skipping.")

df_stat = pd.DataFrame(resultats)
display(df_stat)

print("\n  Interprétation : p < 0.05 → la baisse est statistiquement significative")

In [ ]:
import plotly.graph_objects as go

# Filter for statistically significant pollutants
df_significant_pollutants = df_stat[df_stat['Résultat'] == '✅ Significatif']

fig = go.Figure()

# Add bars for 'Moy. avant'
fig.add_trace(go.Bar(
    x=df_significant_pollutants['Polluant'],
    y=df_significant_pollutants['Moy. avant'],
    name='Moyenne avant ZFE (2016-2020)',
    marker_color='rgb(26, 118, 255)',
    hoverinfo='text',
    hovertext=[
        f"<b>Polluant:</b> {row['Polluant']}<br><b>Moy. avant:</b> {row['Moy. avant']:.2f} µg/m³<br><b>Variation:</b> {row['Variation %']:.1f}%<br><b>p-valeur:</b> {row['p-valeur']:.4f}"
        for index, row in df_significant_pollutants.iterrows()
    ]
))

# Add bars for 'Moy. après'
fig.add_trace(go.Bar(
    x=df_significant_pollutants['Polluant'],
    y=df_significant_pollutants['Moy. après'],
    name='Moyenne après ZFE (2021-2025)',
    marker_color='rgb(55, 83, 109)',
    hoverinfo='text',
    hovertext=[
        f"<b>Polluant:</b> {row['Polluant']}<br><b>Moy. après:</b> {row['Moy. après']:.2f} µg/m³<br><b>Variation:</b> {row['Variation %']:.1f}%<br><b>p-valeur:</b> {row['p-valeur']:.4f}"
        for index, row in df_significant_pollutants.iterrows()
    ]
))

# Update layout
fig.update_layout(
    title_text='Comparaison des Polluants Significatifs Avant et Après ZFE (Marseille)',
    xaxis_title='Polluant',
    yaxis_title='Concentration moyenne (µg/m³)',
    barmode='group',
    template='plotly_white',
    hovermode='closest',
    legend_title_text='Période'
)

fig.show()

### Comparaison de la pollution (NO₂ et PM10) : ZFE vs Hors ZFE (Boxplots)

Ce graphique présente des boxplots comparatifs pour le dioxyde d'azote (NO₂) et les particules PM10. Il permet de visualiser la distribution des concentrations de ces polluants dans les zones ZFE et hors ZFE, et de comparer cette distribution avant (2016-2020) et après (2021-2025) la mise en place de la ZFE. Les boxplots aident à identifier les médianes, les quartiles, la variabilité et les valeurs extrêmes pour chaque catégorie.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# Résultat 3 : Boxplots comparatifs ZFE vs hors ZFE
# ─────────────────────────────────────────────

df_poll["Période"] = df_poll["annee"].apply(lambda x: "Avant ZFE (2016–2020)" if x < 2021 else "Après ZFE (2021–2025)")
df_poll["Zone"]    = df_poll["dans_zfe"].map({1: "ZFE Marseille", 0: "Hors ZFE"})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Comparaison NO₂ et PM10 : ZFE vs Hors ZFE, avant/après 2021",
             fontsize=13, fontweight="bold")

palette = {"Avant ZFE (2016–2020)": "#FF8A65", "Après ZFE (2021–2025)": "#42A5F5"}

for ax, polluant, titre in zip(axes, ["NO2_ugm3", "PM10_ugm3"], ["NO₂ (μg/m³)", "PM10 (μg/m³)"]):
    sns.boxplot(
        data=df_poll, x="Zone", y=polluant, hue="Période",
        palette=palette, ax=ax, width=0.55,
        flierprops=dict(marker=".", alpha=0.3, ms=3)
    )
    ax.set_title(titre, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(titre)
    ax.legend(title="Période", fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("boxplot_zfe_vs_hors_zfe.png", dpi=150, bbox_inches="tight")
plt.show()

### Analyse Spatiale : Carte choroplèthe de la pollution (Matplotlib + Contextily)

Cette carte choroplèthe interactive visualise les niveaux moyens de dioxyde d'azote (NO₂) et de particules PM10 par commune. Elle permet de localiser les zones les plus affectées par la pollution et de voir la délimitation de la ZFE superposée pour une analyse spatiale.

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

# ─────────────────────────────────────────────
# Résultat 4 : Carte choroplèthe Matplotlib avec fond contextily
# ─────────────────────────────────────────────

# Calcul pollution moyenne par commune (utiliser 'inseecom' de df_poll)
df_poll_moy = df_poll.groupby("inseecom")[["NO2_ugm3","PM10_ugm3"]].mean().reset_index()

# Fusionner les données de pollution agrégées avec le GeoDataFrame des communes
gdf_map = gdf_communes.merge(df_poll_moy, left_on="codeinsee", right_on="inseecom", how="left")

# Gérer les valeurs NaN si certaines communes n'ont pas de données de pollution
for col in ["NO2_ugm3", "PM10_ugm3"]:
    if col in gdf_map.columns:
        gdf_map[col] = gdf_map[col].fillna(0) # Remplir NaN avec 0 ou une autre stratégie si préférable

# Reprojection pour contextily (EPSG:3857 requis)
gdf_map_3857 = gdf_map.to_crs(epsg=3857)
gdf_zfe_3857 = gdf_zfe.to_crs(epsg=3857)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Analyse Spatiale : Pollution atmosphérique moyenne (2016–2025)\nMétropole Aix-Marseille-Provence",
             fontsize=14, fontweight="bold")

for ax, col, titre, cmap in zip(
    axes,
    ["NO2_ugm3", "PM10_ugm3"],
    ["NO₂ moyen (μg/m³)", "PM10 moyen (μg/m³)"],
    ["YlOrRd", "YlOrBr"]
):
    gdf_map_3857.plot(
        column=col, cmap=cmap, legend=True,
        legend_kwds={"label": titre, "orientation": "vertical", "shrink": 0.7},
        edgecolor="white", linewidth=0.5, alpha=0.85, ax=ax
    )
    # ZFE en surimpression
    gdf_zfe_3857.boundary.plot(ax=ax, color="blue", linewidth=2.5, linestyle="--", label="ZFE")

    # Fond de carte contextily
    try:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.5)
    except Exception as e:
        print(f"Impossible d'ajouter le fond de carte Contextily: {e}") # Fallback for no connection

    # Determine the pollution threshold for the current pollutant
    seuil_polluant = 0
    if col == "NO2_ugm3":
        seuil_polluant = 40 # OMS limit for NO2
    elif col == "PM10_ugm3":
        seuil_polluant = 20 # OMS limit for PM10

    # Labels communes - show name only if pollution is important
    for _, row in gdf_map_3857.iterrows():
        if row[col] > seuil_polluant: # Check if pollution is above threshold
            if row["Nom_officie"] in communes_dans_zfe['Nom_officie'].tolist():
                ax.annotate(row["Nom_officie"], xy=row.geometry.centroid.coords[0],
                            ha="center", fontsize=7, color="#333", fontweight="bold")
            else:
                 ax.annotate(row["Nom_officie"], xy=row.geometry.centroid.coords[0],
                            ha="center", fontsize=7, color="#333")

    ax.set_title(titre, fontweight="bold")
    ax.set_axis_off()
    ax.legend(loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig("carte_choropleth_pollution_AMP.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Carte sauvegardée : carte_choropleth_pollution_AMP.png")

In [ ]:
import json
import folium
from folium.plugins import TimestampedGeoJson, MiniMap
import pandas as pd
import numpy as np

m3 = folium.Map(location=[43.40, 5.20], zoom_start=10,
                tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
                attr="CartoDB")

# Helper function to assign color based on NO2 value, handling NaN
def no2_to_color(val):
    if pd.isna(val): return "lightgray" # Handle missing values
    if val < 25: return "#1a9641" # Green
    elif val < 35: return "#fdae61" # Orange
    elif val < 45: return "#d7191c" # Red
    else:          return "#7b0000" # Dark Red

# Prepare gdf_communes_simplified: convert datetime columns to string for JSON serialization
gdf_communes_simplified_for_map = gdf_communes_simplified.copy()
for col in gdf_communes_simplified_for_map.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
    gdf_communes_simplified_for_map[col] = gdf_communes_simplified_for_map[col].astype(str)

# Construire les features GeoJSON avec timestamps
features_timestamped = []

for annee in range(2016, 2026):
    # Filter df_poll for the current year
    df_annee_pollution = df_poll[df_poll["annee"] == annee]

    # Aggregate NO2_ugm3 by inseecom (commune) for the current year
    df_annee_pollution_agg = df_annee_pollution.groupby('inseecom')['NO2_ugm3'].mean().reset_index()

    # Make sure 'inseecom' in df_annee_pollution_agg is string to match 'codeinsee' in gdf_communes_simplified
    df_annee_pollution_agg['inseecom'] = df_annee_pollution_agg['inseecom'].astype(str)

    # Merge with simplified commune geometries
    gdf_merged_year = gdf_communes_simplified_for_map.merge(
        df_annee_pollution_agg,
        left_on='codeinsee',
        right_on='inseecom',
        how='left' # Keep all communes, pollution data will be NaN if not found
    )

    # Iterate over the merged GeoDataFrame to create features
    for _, row in gdf_merged_year.iterrows():
        commune_name = row["Nom_officie"]
        val = row["NO2_ugm3"] # This will be NaN if no data for the commune/year

        feature = {
            "type": "Feature",
            "geometry": row.geometry.__geo_interface__,
            "properties": {
                "time": f"{annee}-01-01", # Folium expects YYYY-MM-DD
                "style": {"color": "white", "weight": 1,
                          "fillColor": no2_to_color(val), "fillOpacity": 0.7},
                "popup": f"{commune_name} — NO₂ {annee}: {val:.1f} μg/m³" if pd.notna(val) else f"{commune_name} — NO₂ {annee}: Données non disponibles"
            }
        }
        features_timestamped.append(feature)

TimestampedGeoJson(
    {"type": "FeatureCollection", "features": features_timestamped},
    period="P1Y",
    duration="P1Y",
    transition_time=800,
    auto_play=False,
    loop=False,
    time_slider_drag_update=True
).add_to(m3)

# ZFE
folium.GeoJson(
    gdf_zfe_for_map.__geo_interface__, # Changed to gdf_zfe_for_map
    style_function=lambda x: {"fillColor":"none","color":"blue","weight":3,"dashArray":"6,3"},
    name="ZFE", tooltip="ZFE Marseille"
).add_to(m3)

# Légende couleurs
legend3 = """
<div style="position:fixed;bottom:80px;right:10px;z-index:1000;
background:white;padding:10px;border-radius:8px;border:1px solid #ccc;font-size:12px;">
<b>NO₂ (μg/m³)</b><br>
<span style="background:#1a9641;width:12px;height:12px;display:inline-block;"></span> &lt; 25<br>
<span style="background:#fdae61;width:12px;height:12px;display:inline-block;"></span> 25–35<br>
<span style="background:#d7191c;width:12px;height:12px;display:inline-block;"></span> 35–45<br>
<span style="background:#7b0000;width:12px;height:12px;display:inline-block;"></span> &gt; 45<br>
<span style="background:lightgray;width:12px;height:12px;display:inline-block;"></span> Données non disp.<br>
<span style="background:none;border:2px dashed blue;display:inline-block;width:12px;"></span> ZFE
</div>"""
m3.get_root().html.add_child(folium.Element(legend3))

MiniMap().add_to(m3)
print("✅ Carte animée créée — utilisez le curseur temporel pour naviguer 2016→2025")
m3

Analyse ZFE: Zone d'influence spatiale

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# ─────────────────────────────────────────────
# Résultat 6 : Analyse ZFE — zones d'influence spatiale
# Buffer analysis : pollution en fonction de la distance à la ZFE
# ─────────────────────────────────────────────

# Distance de chaque commune au centroïde ZFE (en km)
# Re-assigning variables from previous cells for clarity and to resolve NameError
gdf_L93 = gdf_communes_L93.copy()
zfe_L93 = gdf_zfe_L93.copy()
zfe_centroid = zfe_L93.geometry.centroid.iloc[0]

gdf_L93["dist_zfe_km"] = gdf_L93.geometry.centroid.distance(zfe_centroid) / 1000

# Join pollution data
# Use df_poll_moy which contains 'inseecom' and 'NO2_ugm3'
# Merge with gdf_L93 using 'codeinsee' from gdf_L93 and 'inseecom' from df_poll_moy
gdf_dist = gdf_L93.merge(df_poll_moy, left_on="codeinsee", right_on="inseecom", how="left")

# Drop the redundant 'inseecom' column if it exists after merge
if 'inseecom' in gdf_dist.columns:
    gdf_dist = gdf_dist.drop(columns=['inseecom'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Relation entre distance à la ZFE et pollution (NO₂) — Métropole AMP",
             fontsize=13, fontweight="bold")

# Scatter : distance vs NO2
ax = axes[0]
scatter = ax.scatter(
    gdf_dist["dist_zfe_km"], gdf_dist["NO2_ugm3"],
    c=gdf_dist["NO2_ugm3"], cmap="YlOrRd", s=80, zorder=3, edgecolors="white"
)

# Annotate communes. Use 'Nom_officie' from gdf_communes (now in gdf_dist)
for _, row in gdf_dist.iterrows():
    # Only annotate if NO2 data is available
    if pd.notna(row["NO2_ugm3"]):
        ax.annotate(row["Nom_officie"], (row["dist_zfe_km"]+0.1, row["NO2_ugm3"]+0.2), fontsize=8)

# Régression linéaire
valid = gdf_dist.dropna(subset=["dist_zfe_km","NO2_ugm3"])
if not valid.empty:
    z = np.polyfit(valid["dist_zfe_km"], valid["NO2_ugm3"], 1)
    xfit = np.linspace(0, valid["dist_zfe_km"].max(), 100)
    ax.plot(xfit, np.polyval(z, xfit), "--", color="steelblue", lw=1.5, label=f"Régression (pente={z[0]:.2f})")
else:
    print("Warning: Not enough data for linear regression on distance vs NO2.")

ax.set_xlabel("Distance à la ZFE (km)")
ax.set_ylabel("NO₂ moyen (μg/m³)")
ax.set_title("Distance ZFE vs NO₂ moyen")
ax.legend()
ax.grid(alpha=0.3)
plt.colorbar(scatter, ax=ax, label="NO₂ (μg/m³)")

# Bar chart : communes triées par NO2
ax2 = axes[1]
# Drop rows with NaN NO2_ugm3 for sorting and plotting
gdf_sorted = gdf_dist.dropna(subset=["NO2_ugm3"]).sort_values("NO2_ugm3", ascending=True)

# Get the list of ZFE commune names from the previously identified 'communes_dans_zfe'
zfe_communes_names = communes_dans_zfe['Nom_officie'].tolist()

# Dynamically assign colors based on whether the commune is in the ZFE
colors = ["#E53935" if nom_commune in zfe_communes_names else "#42A5F5" for nom_commune in gdf_sorted["Nom_officie"]]

ax2.barh(gdf_sorted["Nom_officie"], gdf_sorted["NO2_ugm3"], color=colors, edgecolor="white")
ax2.axvline(x=40, color="red", lw=1.5, ls="--", label="Seuil OMS (40 μg/m³)")
ax2.set_xlabel("NO₂ moyen (μg/m³)")
ax2.set_title("Classement communes par niveau NO₂")
ax2.legend(fontsize=9)
ax2.grid(axis="x", alpha=0.3)

# Update legend for the bar chart
ax2.legend(handles=[
    Patch(color="#E53935", label="Communes dans ZFE"),
    Patch(color="#42A5F5", label="Communes hors ZFE"),
    plt.Line2D([0],[0], color="red", ls="--", label="Seuil OMS")
], fontsize=9)

plt.tight_layout()
plt.savefig("analyse_spatiale_distance_zfe.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# Résultat 7 : Carte Plotly Choroplèthe — Bilan 2016 vs 2025
# ─────────────────────────────────────────────

# Filter for 2016 and 2025 data, grouping by 'inseecom' (commune identifier)
df_2016 = df_poll[df_poll["annee"]==2016].groupby("inseecom")["NO2_ugm3"].mean().reset_index()
df_2025 = df_poll[df_poll["annee"]==2025].groupby("inseecom")["NO2_ugm3"].mean().reset_index()

# Merge the 2016 and 2025 dataframes on 'inseecom'
df_diff = df_2016.merge(df_2025, on="inseecom", suffixes=('_2016','_2025'))

# Calculate percentage variation
df_diff["variation_pct"] = (df_diff["NO2_ugm3_2025"] - df_diff["NO2_ugm3_2016"]) / df_diff["NO2_ugm3_2016"] * 100

print("📊 Bilan de l'évolution du NO₂ par commune (2016 → 2025) :")
# Display relevant columns, using 'inseecom' as the commune identifier
print(df_diff[["inseecom","NO2_ugm3_2016","NO2_ugm3_2025","variation_pct"]]
      .sort_values("variation_pct")
      .rename(columns={"inseecom":"INSEE Comm.", "NO2_ugm3_2016":"NO₂ 2016","NO2_ugm3_2025":"NO₂ 2025","variation_pct":"Δ %"})
      .round(2)
      .to_string(index=False))

In [ ]:
import plotly.graph_objects as go

# --- 1. Préparer les données pour la carte ---
# Fusionner les données de variation de pollution avec le GeoDataFrame des communes
gdf_communes_diff = gdf_communes.merge(df_diff, left_on='codeinsee', right_on='inseecom', how='left')

# Gérer les NaN pour la colonne de variation_pct pour la visualisation (par exemple, les mettre à 0 ou NaN pour qu'ils soient grisés)
# Pour une variation en %, NaN est plus approprié pour indiquer l'absence de données valides pour le calcul.
gdf_communes_diff['variation_pct'] = gdf_communes_diff['variation_pct'].fillna(np.nan)

# --- 2. Créer la carte Choroplèthe Plotly ---
fig = go.Figure(
    go.Choroplethmapbox(
        geojson=gdf_communes_diff.__geo_interface__,
        locations=gdf_communes_diff.codeinsee,
        z=gdf_communes_diff.variation_pct,
        colorscale="RdBu", # Utilise une échelle de couleurs divergente
        zmin=-50, # Ajustez ces valeurs selon la plage de vos données
        zmax=50,  # Pour centrer l'échelle sur 0, -50% à +50%
        marker_opacity=0.7,
        marker_line_width=1,
        colorbar_title="NO₂ Variation % (2016-2025)",
        # Texte au survol (tooltip)
        hovertemplate=
            "<b>Commune:</b> %{customdata[0]}<br>" +
            "<b>NO₂ 2016:</b> %{customdata[1]:.1f} µg/m³<br>" +
            "<b>NO₂ 2025:</b> %{customdata[2]:.1f} µg/m³<br>" +
            "<b>Variation:</b> %{customdata[3]:.1f}%<extra></extra>",
        customdata=np.stack((gdf_communes_diff.Nom_officie,
                             gdf_communes_diff.NO2_ugm3_2016,
                             gdf_communes_diff.NO2_ugm3_2025,
                             gdf_communes_diff.variation_pct), axis=-1)
    )
)

# --- 3. Ajouter la couche ZFE (GeoJSON) ---
fig.add_trace(
    go.Choroplethmapbox(
        geojson=gdf_zfe_for_map.__geo_interface__, # Utilise le GeoDataFrame ZFE préparé
        locations=gdf_zfe_for_map.index, # Ou un identifiant unique si disponible
        z=[1]*len(gdf_zfe_for_map), # Une valeur constante pour juste montrer la géométrie
        colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']], # Totalement transparent
        showscale=False,
        marker_line_color='blue',
        marker_line_width=3,
        marker_opacity=0, # Rendre l'intérieur transparent
        name='ZFE Marseille',
        hovertemplate='<b>ZFE Marseille</b><extra></extra>'
    )
)

# --- 4. Mise en page de la carte ---
fig.update_layout(
    title_text="Carte Choroplèthe : Variation du NO₂ par commune (2016 vs 2025)",
    mapbox_style="carto-positron", # Fond de carte
    mapbox_zoom=9, # Zoom initial
    mapbox_center={"lat": centre_AMP[0], "lon": centre_AMP[1]}, # Centrage sur la métropole
    margin={"r":0,"t":50,"l":0,"b":0},
    height=600,
    legend_title_text='Légende'
)

fig.show()
print("✅ Carte Choroplèthe Plotly de la variation de NO₂ générée.")

### Profil Mensuel (Saisonnalité) du NO₂

Ce graphique présente l'évolution moyenne du dioxyde d'azote (NO₂) par mois, comparant les communes situées dans la Zone à Faibles Émissions (ZFE) de Marseille et celles situées hors de la ZFE. Cela permet d'identifier les tendances saisonnières et les différences de pollution entre les deux types de zones.

In [ ]:
# ─────────────────────────────────────────────
# Résultat 8 : Graphique Plotly — Profil mensuel (saisonnalité)
# ─────────────────────────────────────────────

df_saison = df_poll.groupby(["mois","Zone"])["NO2_ugm3"].mean().reset_index()

mois_noms = {1:"Jan",2:"Fév",3:"Mar",4:"Avr",5:"Mai",6:"Jun",
             7:"Jul",8:"Aoû",9:"Sep",10:"Oct",11:"Nov",12:"Déc"}
df_saison["mois_nom"] = df_saison["mois"].map(mois_noms)

fig_saison = px.line(
    df_saison, x="mois_nom", y="NO2_ugm3", color="Zone",
    markers=True,
    color_discrete_map={"ZFE Marseille": "#E53935", "Hors ZFE": "#1976D2"},
    title="Profil saisonnier du NO₂ — ZFE vs Hors ZFE (moyenne 2016–2025)",
    labels={"NO2_ugm3": "NO₂ (μg/m³)", "mois_nom": "Mois", "Zone": "Zone"}
)
fig_saison.update_layout(template="plotly_white", height=400)
fig_saison.add_hline(y=40, line_dash="dash", line_color="gray",
                      annotation_text="Seuil OMS 40 μg/m³")
fig_saison.show()

### Niveaux de Pollution Moyens par Polluant et par Commune

Ce graphique présente les niveaux moyens des principaux polluants (NO₂, PM₁₀, PM₂.₅) pour chaque commune. Il permet de comparer visuellement les concentrations de polluants entre les différentes communes et d'identifier celles qui sont le plus affectées par certains types de pollution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure 'inseecom' in df_poll_moy is string to match 'codeinsee' in gdf_communes
df_poll_moy['inseecom'] = df_poll_moy['inseecom'].astype(str)

# Merge df_poll_moy with gdf_communes to get commune names (Nom_officie)
df_commune_pollution_avg = gdf_communes.merge(
    df_poll_moy,
    left_on='codeinsee',
    right_on='inseecom',
    how='left'
)

# Select relevant columns for plotting
# Make sure 'PM25_ugm3' exists, otherwise handle it (fillna(0) was applied earlier)
pollutant_columns = ['NO2_ugm3', 'PM10_ugm3', 'PM25_ugm3']

# Melt the DataFrame to a long format for easier plotting with seaborn
pollutant_avg_melted = df_commune_pollution_avg.melt(
    id_vars=['Nom_officie'], # Use commune official name
    value_vars=[col for col in pollutant_columns if col in df_commune_pollution_avg.columns], # Only include existing columns
    var_name='Pollutant_Type',
    value_name='Average_Value'
)

# Clean up Pollutant_Type names for better readability in the plot
pollutant_avg_melted['Pollutant_Type'] = pollutant_avg_melted['Pollutant_Type'].str.replace('_ugm3', '')

# Drop rows where Average_Value is NaN (communes with no data for a specific pollutant)
pollutant_avg_melted = pollutant_avg_melted.dropna(subset=['Average_Value'])

# Plotting the grouped bar chart
plt.figure(figsize=(16, 9))
sns.barplot(x='Nom_officie', y='Average_Value', hue='Pollutant_Type', data=pollutant_avg_melted, palette='tab10')

plt.title('Average Pollution Levels by Pollutant Type per Commune', fontsize=18)
plt.xlabel('Commune Name', fontsize=14)
plt.ylabel('Average Pollution Value (μg/m³)', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.legend(title='Pollutant Type', title_fontsize='12', fontsize='10')
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd # Ensure pandas is imported if not already

# 1. Create no2_pollution_gdf from df_raw_pollution
# Filter for NO2 pollutant
no2_raw_data = df_raw_pollution[df_raw_pollution['label_polluant'] == "Dioxyde d'azote (gaz)"].copy()

# Drop rows where 'geo_point_2d' is NaN or empty to avoid errors during splitting
no2_raw_data.dropna(subset=['geo_point_2d'], inplace=True)

# Extract latitude and longitude from 'geo_point_2d' string
# The 'geo_point_2d' column is a dictionary-like object, need to access 'lat' and 'lon'
no2_raw_data['latitude'] = no2_raw_data['geo_point_2d'].apply(lambda x: x['lat'] if isinstance(x, dict) and 'lat' in x else None)
no2_raw_data['longitude'] = no2_raw_data['geo_point_2d'].apply(lambda x: x['lon'] if isinstance(x, dict) and 'lon' in x else None)

# Drop rows where latitude or longitude extraction failed
no2_raw_data.dropna(subset=['latitude', 'longitude'], inplace=True)

# Create Point geometries
geometry = [Point(xy) for xy in zip(no2_raw_data['longitude'], no2_raw_data['latitude'])]

# Create the GeoDataFrame for NO2 pollution points
no2_pollution_gdf = gpd.GeoDataFrame(no2_raw_data, geometry=geometry, crs="EPSG:4326")

# 2. Use existing GeoDataFrames for metropolitan limits and ZFE
communes_gdf = gdf_communes.copy() # Already in EPSG:4326
zfe_gdf = gdf_zfe.copy()       # Already in EPSG:4326

# Create the base plot
fig, ax = plt.subplots(1, 1, figsize=(15, 15))

# Plot metropolitan limits as background for context
communes_gdf.plot(ax=ax, color='lightgray', edgecolor='black', label='Limites Métropolitaines')

# Plot ZFE areas for context
zfe_gdf.plot(ax=ax, color='orange', alpha=0.3, edgecolor='red', label='ZFE')

# Check if no2_pollution_gdf has enough points for KDE
if len(no2_pollution_gdf) > 1:
    # Extract x (longitude) and y (latitude) coordinates from NO2 pollution points
    x_coords = no2_pollution_gdf.geometry.x
    y_coords = no2_pollution_gdf.geometry.y

    # Plot KDE of NO2 pollution points
    sns.kdeplot(
        x=x_coords, y=y_coords,
        cmap='hot_r', # Reverse hot colormap for pollution intensity
        fill=True,
        alpha=0.6,
        levels=10, # Number of contour levels
        ax=ax,
        cbar=True,  # Add color bar for density
        cbar_kws={'label': 'Densité de points de mesure NO₂'} # Label for the color bar
    )
else:
    print("Warning: Not enough NO2 pollution points to generate KDE plot. Showing only point locations.")

# Add original NO2 pollution points for reference
no2_pollution_gdf.plot(ax=ax, marker='o', color='blue', markersize=20, alpha=0.7, label='Points de Mesure NO2', zorder=5)

# Add title and labels
ax.set_title('Analyse de Densité Spatiale des Niveaux de Pollution NO2', fontsize=16)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Create custom legend handles
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

metropolitan_limits_handle = mpatches.Patch(facecolor='lightgray', edgecolor='black', label='Limites Métropolitaines')
zfe_handle = mpatches.Patch(facecolor='orange', alpha=0.3, edgecolor='red', label='ZFE')
no2_points_handle = mlines.Line2D([], [], color='blue', marker='o', linestyle='None', markersize=8, label='Points de Mesure NO2')

# Create a dummy legend entry for the KDE (heatmap itself) if it was plotted
if len(no2_pollution_gdf) > 1:
    kde_handle = mpatches.Patch(color='red', alpha=0.6, label='Densité NO2 (KDE)')
    ax.legend(handles=[metropolitan_limits_handle, zfe_handle, no2_points_handle, kde_handle], loc='upper left', fontsize=10)
else:
    ax.legend(handles=[metropolitan_limits_handle, zfe_handle, no2_points_handle], loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# 1. Create pm10_pollution_gdf from df_raw_pollution
# Filter for PM10 pollutant
pm10_raw_data = df_raw_pollution[df_raw_pollution['label_polluant'] == "Particules en suspension <10 µm (masses) (PM10)"].copy()

# Drop rows where 'geo_point_2d' is NaN or empty
pm10_raw_data.dropna(subset=['geo_point_2d'], inplace=True)

# Extract latitude and longitude from 'geo_point_2d' string
# The 'geo_point_2d' column is a dictionary-like object, need to access 'lat' and 'lon'
pm10_raw_data['latitude'] = pm10_raw_data['geo_point_2d'].apply(lambda x: x['lat'] if isinstance(x, dict) and 'lat' in x else None)
pm10_raw_data['longitude'] = pm10_raw_data['geo_point_2d'].apply(lambda x: x['lon'] if isinstance(x, dict) and 'lon' in x else None)

# Drop rows where latitude or longitude extraction failed
pm10_raw_data.dropna(subset=['latitude', 'longitude'], inplace=True)

# Create Point geometries
geometry_pm10 = [Point(xy) for xy in zip(pm10_raw_data['longitude'], pm10_raw_data['latitude'])]

# Create the GeoDataFrame for PM10 pollution points
pm10_pollution_gdf = gpd.GeoDataFrame(pm10_raw_data, geometry=geometry_pm10, crs="EPSG:4326")

# 2. Use existing GeoDataFrames for metropolitan limits and ZFE
communes_gdf_pm10 = gdf_communes.copy() # Already in EPSG:4326
zfe_gdf_pm10 = gdf_zfe.copy()       # Already in EPSG:4326

# Create the base plot
fig_pm10, ax_pm10 = plt.subplots(1, 1, figsize=(15, 15))

# Plot metropolitan limits as background for context
communes_gdf_pm10.plot(ax=ax_pm10, color='lightgray', edgecolor='black', label='Limites Métropolitaines')

# Plot ZFE areas for context
zfe_gdf_pm10.plot(ax=ax_pm10, color='orange', alpha=0.3, edgecolor='red', label='ZFE')

# Check if pm10_pollution_gdf has enough points for KDE
if len(pm10_pollution_gdf) > 1:
    # Extract x (longitude) and y (latitude) coordinates from PM10 pollution points
    x_coords_pm10 = pm10_pollution_gdf.geometry.x
    y_coords_pm10 = pm10_pollution_gdf.geometry.y

    # Plot KDE of PM10 pollution points
    sns.kdeplot(
        x=x_coords_pm10, y=y_coords_pm10,
        cmap='Blues_r', # Reverse blue colormap for PM10 intensity
        fill=True,
        alpha=0.6,
        levels=10, # Number of contour levels
        ax=ax_pm10,
        cbar=True, # Add color bar for density
        cbar_kws={'label': 'Densité de points de mesure PM10'} # Label for the color bar
    )
else:
    print("Warning: Not enough PM10 pollution points to generate KDE plot. Showing only point locations.")

# Add original PM10 pollution points for reference
pm10_pollution_gdf.plot(ax=ax_pm10, marker='s', color='darkblue', markersize=20, alpha=0.7, label='Points de Mesure PM10', zorder=5)

# Add title and labels
ax_pm10.set_title('Analyse de Densité Spatiale des Niveaux de Pollution PM10', fontsize=16)
ax_pm10.set_xlabel('Longitude')
ax_pm10.set_ylabel('Latitude')

# Create custom legend handles
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

metropolitan_limits_handle_pm10 = mpatches.Patch(facecolor='lightgray', edgecolor='black', label='Limites Métropolitaines')
zfe_handle_pm10 = mpatches.Patch(facecolor='orange', alpha=0.3, edgecolor='red', label='ZFE')
pm10_points_handle = mlines.Line2D([], [], color='darkblue', marker='s', linestyle='None', markersize=8, label='Points de Mesure PM10')

# Create a dummy legend entry for the KDE (heatmap itself) if it was plotted
if len(pm10_pollution_gdf) > 1:
    kde_handle_pm10 = mpatches.Patch(color='blue', alpha=0.6, label='Densité PM10 (KDE)')
    ax_pm10.legend(handles=[metropolitan_limits_handle_pm10, zfe_handle_pm10, pm10_points_handle, kde_handle_pm10], loc='upper left', fontsize=10)
else:
    ax_pm10.legend(handles=[metropolitan_limits_handle_pm10, zfe_handle_pm10, pm10_points_handle], loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# 1. Create pm25_pollution_gdf from df_raw_pollution
# Filter for PM2.5 pollutant using the correct label from df_raw_pollution
pm25_raw_data = df_raw_pollution[df_raw_pollution['label_polluant'] == "Particules en suspension <2.5 µm (masses) (PM2.5)"].copy()

# Drop rows where 'geo_point_2d' is NaN or empty
pm25_raw_data.dropna(subset=['geo_point_2d'], inplace=True)

# Extract latitude and longitude from 'geo_point_2d' string
pm25_raw_data['latitude'] = pm25_raw_data['geo_point_2d'].apply(lambda x: x['lat'] if isinstance(x, dict) and 'lat' in x else None)
pm25_raw_data['longitude'] = pm25_raw_data['geo_point_2d'].apply(lambda x: x['lon'] if isinstance(x, dict) and 'lon' in x else None)

# Drop rows where latitude or longitude extraction failed
pm25_raw_data.dropna(subset=['latitude', 'longitude'], inplace=True)

# Create Point geometries
geometry_pm25 = [Point(xy) for xy in zip(pm25_raw_data['longitude'], pm25_raw_data['latitude'])]

# Create the GeoDataFrame for PM2.5 pollution points
pm25_pollution_gdf = gpd.GeoDataFrame(pm25_raw_data, geometry=geometry_pm25, crs="EPSG:4326")

# 2. Use existing GeoDataFrames for metropolitan limits and ZFE
communes_gdf_pm25 = gdf_communes.copy() # Already in EPSG:4326
zfe_gdf_pm25 = gdf_zfe.copy()       # Already in EPSG:4326

# Create the base plot
fig_pm25, ax_pm25 = plt.subplots(1, 1, figsize=(15, 15))

# Plot metropolitan limits as background for context
communes_gdf_pm25.plot(ax=ax_pm25, color='lightgray', edgecolor='black', label='Limites Métropolitaines')

# Plot ZFE areas for context
zfe_gdf_pm25.plot(ax=ax_pm25, color='orange', alpha=0.3, edgecolor='red', label='ZFE')

# Check if pm25_pollution_gdf has enough points for KDE
if len(pm25_pollution_gdf) > 1:
    # Extract x (longitude) and y (latitude) coordinates from PM2.5 pollution points
    x_coords_pm25 = pm25_pollution_gdf.geometry.x
    y_coords_pm25 = pm25_pollution_gdf.geometry.y

    # Plot KDE of PM2.5 pollution points
    sns.kdeplot(
        x=x_coords_pm25, y=y_coords_pm25,
        cmap='Greens_r', # Reverse green colormap for PM2.5 intensity
        fill=True,
        alpha=0.6,
        levels=10, # Number of contour levels
        ax=ax_pm25,
        cbar=True, # Add color bar for density
        cbar_kws={'label': 'Densité de points de mesure PM2.5'} # Label for the color bar
    )
else:
    print("Warning: Not enough PM2.5 pollution points to generate KDE plot. Showing only point locations.")

# Add original PM2.5 pollution points for reference
pm25_pollution_gdf.plot(ax=ax_pm25, marker='^', color='darkgreen', markersize=20, alpha=0.7, label='Points de Mesure PM2.5', zorder=5)

# Add title and labels
ax_pm25.set_title('Analyse de Densité Spatiale des Niveaux de Pollution PM2.5', fontsize=16)
ax_pm25.set_xlabel('Longitude')
ax_pm25.set_ylabel('Latitude')

# Create custom legend handles
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

metropolitan_limits_handle_pm25 = mpatches.Patch(facecolor='lightgray', edgecolor='black', label='Limites Métropolitaines')
zfe_handle_pm25 = mpatches.Patch(facecolor='orange', alpha=0.3, edgecolor='red', label='ZFE')
pm25_points_handle = mlines.Line2D([], [], color='darkgreen', marker='^', linestyle='None', markersize=8, label='Points de Mesure PM2.5')

# Create a dummy legend entry for the KDE (heatmap itself) if it was plotted
if len(pm25_pollution_gdf) > 1:
    kde_handle_pm25 = mpatches.Patch(color='green', alpha=0.6, label='Densité PM2.5 (KDE)')
    ax_pm25.legend(handles=[metropolitan_limits_handle_pm25, zfe_handle_pm25, pm25_points_handle, kde_handle_pm25], loc='upper left', fontsize=10)
else:
    ax_pm25.legend(handles=[metropolitan_limits_handle_pm25, zfe_handle_pm25, pm25_points_handle], loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()